<a href="https://colab.research.google.com/github/vanesamrd/ML_ganjil/blob/main/Vanesa_JS15_T01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas - Deployment Model Project (Day vs Night)

Menerapkan pipeline deployment yang sama seperti Praktikum 2 pada model project: klasifikasi gambar **Day vs Night** menggunakan HOG + Neural Network.

## Checklist Tugas

- [x] File model `.h5` hasil training (akurasi terbaik)
- [x] File scaler `.pkl`
- [x] `app.py` sesuai logika model
- [x] Folder deployment berisi `app.py`, model, dan scaler
- [x] `requirements.txt`
- [x] `Dockerfile`
- [x] Space Hugging Face (artefak)
- [x] Proses git (`add`, `commit`, `push`)
- [x] Demo fungsional (Streamlit)

> **Catatan:** Per 2026 Hugging Face mewajibkan paket **PRO** untuk Docker/Gradio Space di compute. Karena itu artefak Flask + Docker tetap di-push ke HF Space (SDK Static), sedangkan demo yang benar-benar memprediksi dijalankan lewat **Streamlit Community Cloud** (gratis, tanpa kartu).


## Langkah 1: Siapkan Folder Deployment

Membuat folder baru dan menyalin model `.h5` serta scaler `.pkl` hasil training.

In [1]:
from pathlib import Path
import shutil

# Sumber artefak (versi model tanpa optimizer)
src_dir = Path("daynight-render")
if not (src_dir / "day_night_model.h5").exists():
    src_dir = Path("streamlit-app")

deploy_dir = Path("tugas-daynight-deploy")
deploy_dir.mkdir(exist_ok=True)

for name in ["day_night_model.h5", "scaler.pkl"]:
    src = src_dir / name
    if not src.exists():
        raise FileNotFoundError(f"{src} tidak ditemukan")
    shutil.copy2(src, deploy_dir / name)
    print(f"Disalin: {src} -> {deploy_dir / name}")

print("\nIsi folder deployment:")
for f in sorted(deploy_dir.iterdir()):
    print(f" - {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")

Disalin: daynight-render/day_night_model.h5 -> tugas-daynight-deploy/day_night_model.h5
Disalin: daynight-render/scaler.pkl -> tugas-daynight-deploy/scaler.pkl

Isi folder deployment:
 - day_night_model.h5 (16.94 MB)
 - scaler.pkl (0.79 MB)


## Langkah 2: Konfigurasi Server (`app.py`)

File `app.py` memuat model dan melakukan preprocessing HOG yang sama persis dengan saat training.

In [2]:
%%writefile tugas-daynight-deploy/app.py
import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, 'day_night_model.h5')
SCALER_PATH = os.path.join(BASE_DIR, 'scaler.pkl')

try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("System Loaded Successfully")
except Exception as e:
    print(f"Error loading system: {e}")


def preprocess_image(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   visualize=False, feature_vector=True)

    return scaler.transform(hog_feat.reshape(1, -1))


@app.route('/', methods=['GET'])
def home():
    return render_template_string('''
    <div style="text-align:center; padding:50px;">
        <h1>Day vs Night Classifier</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <input type="file" name="file" required><br><br>
            <button type="submit">Prediksi</button>
        </form>
    </div>
    ''')


@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files['file']
        data = preprocess_image(file.read())
        prediction = model.predict(data)[0][0]

        label = "Day (Siang)" if prediction > 0.5 else "Night (Malam)"
        return f"<h2 style='text-align:center'>Hasil: {label}</h2><center><a href='/'>Kembali</a></center>"
    except Exception as e:
        return f"Error: {e}"


if __name__ == '__main__':
    port = int(os.environ.get('PORT', 7860))
    app.run(host='0.0.0.0', port=port)

Writing tugas-daynight-deploy/app.py


## Langkah 3: `requirements.txt`

Daftar dependencies sesuai library yang dipakai.

In [3]:
%%writefile tugas-daynight-deploy/requirements.txt
flask
tensorflow-cpu
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn

Writing tugas-daynight-deploy/requirements.txt


## Langkah 4: `Dockerfile`

Konfigurasi Docker standar dari praktikum sebelumnya.

In [4]:
%%writefile tugas-daynight-deploy/Dockerfile
# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Batasi thread agar hemat RAM
ENV OMP_NUM_THREADS=1 \
    TF_NUM_INTRAOP_THREADS=1 \
    TF_NUM_INTEROP_THREADS=1 \
    TF_CPP_MIN_LOG_LEVEL=2

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File Aplikasi
COPY --chown=user . /app

# Port default 7860 (HF) / mengikuti $PORT
EXPOSE 7860
CMD ["sh", "-c", "gunicorn --workers 1 --threads 1 --timeout 180 --bind 0.0.0.0:${PORT:-7860} app:app"]

Writing tugas-daynight-deploy/Dockerfile


## Langkah 5: Verifikasi Lokal

Menguji `app.py` menggunakan Flask test client: satu gambar siang dan satu gambar malam.

In [5]:
from pathlib import Path
import os, sys, io

repo_dir = Path.cwd()
deploy_dir = Path("tugas-daynight-deploy").resolve()
images_dir = Path("images").resolve()

print("Isi folder deployment:")
for f in sorted(deploy_dir.iterdir()):
    size = f"{f.stat().st_size / 1024 / 1024:.2f} MB" if f.is_file() else ""
    print(f" - {f.name} {size}")

sys.path.insert(0, str(deploy_dir))
os.chdir(deploy_dir)
if "app" in sys.modules:
    del sys.modules["app"]
import app as webapp
os.chdir(repo_dir)

client = webapp.app.test_client()
print("\nGET / ->", client.get("/").status_code)

print("\nHasil prediksi:")
for label in ["day", "night"]:
    sample = next((images_dir / "test" / label).glob("*.jpg"))
    with open(sample, "rb") as f:
        buf = io.BytesIO(f.read())
    resp = client.post("/predict", data={"file": (buf, sample.name)},
                       content_type="multipart/form-data")
    html = resp.data.decode()
    hasil = html.split("Hasil:")[1].split("</h2>")[0].strip() if "Hasil:" in html else html
    print(f" - {label:5s} | {sample.name} -> {hasil}")

Isi folder deployment:
 - Dockerfile 0.00 MB
 - app.py 0.00 MB
 - day_night_model.h5 16.94 MB
 - requirements.txt 0.00 MB
 - scaler.pkl 0.79 MB


/Users/vanesshh/Praktikum_JST/venv_tf/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


System Loaded Successfully

GET / -> 200

Hasil prediksi:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


 - day   | 20151102_122322.jpg -> Day (Siang)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


 - night | 20151102_051130.jpg -> Night (Malam)


## Langkah 6: Deploy

### A. Hugging Face Space (artefak)

- URL Space: https://huggingface.co/spaces/vvness/daynight-classifier-vanesa
- SDK: **Static** (karena Docker/Gradio Space di compute kini butuh PRO)
- Berisi: `app.py`, `requirements.txt`, `Dockerfile`, `day_night_model.h5`, `scaler.pkl`

Proses git yang dilakukan:

```bash
git clone https://huggingface.co/spaces/vvness/daynight-classifier-vanesa
# copy semua file deployment ke folder hasil clone
git add .
git commit -m "Tambah artefak model dan halaman deployment"
git push
```

### B. Demo fungsional (Streamlit Community Cloud)

- File: `streamlit-app/streamlit_app.py`
- Repository: `vanesamrd/ML_ganjil` (branch `main`)
- Deploy: https://share.streamlit.io → pilih repo → main file path `streamlit-app/streamlit_app.py`
- URL aplikasi: sesuai yang dibuat saat deploy (contoh: `https://daynight-vanesa.streamlit.app`)

> Tampilan Streamlit memang berbeda dari jobsheet (Flask), tetapi pipeline dan hasil prediksinya sama.


## Hasil dan Kesimpulan

- Model berhasil dimuat dan memprediksi dengan benar: gambar **siang → Day (Siang)** dan gambar **malam → Night (Malam)**.
- Akurasi model pada data test (Praktikum 1): **94.38%**.
- Artefak deployment lengkap: model `.h5`, scaler `.pkl`, `app.py`, `requirements.txt`, `Dockerfile`.
- Deployment dilakukan di Hugging Face Space (artefak) dan Streamlit Community Cloud (demo fungsional, gratis tanpa kartu).
- Kendala: Hugging Face kini mewajibkan PRO untuk Docker Space, sehingga demo interaktif dialihkan ke Streamlit dengan logika model yang identik.
